# D2 · Validation and learning curves: diagnosing a model

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Compute a validation curve (depth of a tree) and two learning curves (logistic regression and random forest), and use them
to say whether a model suffers from underfitting or overfitting and whether more data would be needed.

## Dataset

`ds.loans(with_missing=False)`: the four numeric columns and the target `defaulted`, 1500 rows. Stratified 5-fold cross-validation and
AUC scoring. Models: `DecisionTreeClassifier` (for the validation curve), `LogisticRegression` and `RandomForestClassifier(60)`.

## Educational purpose

Learn to read the two plots that answer the two most useful questions of a project: "is the model too simple or too
complex?" (validation curve) and "would adding data help?" (learning curve). The answers are computed, not guessed.

## Task

1. With `validation_curve` compute the training and validation AUC of the tree for `max_depth` in `[1, 2, 3, 5, 8, 12, 20]` and save `train_v`, `val_v` (means over the folds, array of 7 values), `best_depth` (the value with the maximum validation) and `gap_v = train_v − val_v`.
2. Assign to `diagnosis` a dictionary with three keys (`"depth_1"`, `"depth_20"`, `"depth_best"`) and values `"underfitting"`, `"overfitting"` or `"adequate"` applying the rules: **overfitting** if the gap exceeds 0.15; **underfitting** if the validation is below 0.72; otherwise **adequate**.
3. With `learning_curve` (`train_sizes = [0.2, 0.4, 0.6, 0.8, 1.0]`) compute for the logistic and for the forest the validation means at each size: `val_lr` and `val_rf` (arrays of 5 values).
4. Save `lr_gain` and `rf_gain`: the validation AUC with 100% of the data minus the one with 60% (how much you would still gain by adding data). Assign to `more_data_needed` a dictionary `{"lr": bool, "rf": bool}`: `True` if the gain exceeds 0.01.
5. Save `sufficient_fraction`, a dictionary `{"lr": ..., "rf": ...}` with the **minimum** fraction of data (among `sizes`) for which the validation AUC is within 0.005 of the one obtained with all the data.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, learning_curve, validation_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y = df["defaulted"].to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
depths = [1, 2, 3, 5, 8, 12, 20]

## Your work

In [ ]:
tr, va = validation_curve(DecisionTreeClassifier(random_state=0), X, y, param_name="max_depth", param_range=depths,
                          cv=cv, scoring="roc_auc")
train_v, val_v = ..., ...
best_depth = ...
gap_v = ...
pd.DataFrame({"max_depth": depths, "train": train_v, "validation": val_v, "gap": gap_v}).round(3)

In [ ]:
diagnosis = {"depth_1": ..., "depth_20": ..., "depth_best": ...}

In [ ]:
sizes = [0.2, 0.4, 0.6, 0.8, 1.0]
_, _, va_lr = learning_curve(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X, y, train_sizes=sizes, cv=cv, scoring="roc_auc")
_, _, va_rf = learning_curve(RandomForestClassifier(60, min_samples_leaf=5, random_state=0), X, y, train_sizes=sizes, cv=cv, scoring="roc_auc")
val_lr, val_rf = ..., ...
lr_gain = ...
rf_gain = ...
more_data_needed = {"lr": ..., "rf": ...}
sufficient_fraction = {"lr": ..., "rf": ...}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`tr.mean(axis=1)` and `va.mean(axis=1)`. For the learning curve: `va_lr.mean(axis=1)`. `more_data_needed = {"lr": bool(lr_gain > 0.01), "rf": bool(rf_gain > 0.01)}`. A rule for the diagnosis: `gap = train - val`; if `gap > 0.15` overfitting, elif `val < 0.72` underfitting, otherwise adequate.'):
    check(len(train_v) == 7 and len(val_v) == 7 and best_depth == depths[int(np.argmax(val_v))], "validation curve computed, best_depth consistent", "best_depth = depths[np.argmax(val_v)]")
    check(2 <= best_depth <= 8, "the best depth is intermediate (between 2 and 8): neither too little nor too much", "look at the validation: it goes up and then down")
    check(gap_v[-1] > 0.15 and gap_v[0] < 0.05 and close(gap_v, train_v - val_v), "the gap grows with the depth (tiny at 1, huge at 20)", "gap_v = train_v - val_v")
    check(diagnosis == {"depth_1": "underfitting", "depth_20": "overfitting", "depth_best": "adequate"}, "correct diagnosis for the three trees",
          "depth_1: low validation (underfitting); depth_20: huge gap (overfitting); the best: adequate")
    check(len(val_lr) == 5 and len(val_rf) == 5 and close(lr_gain, val_lr[-1] - val_lr[2]) and close(rf_gain, val_rf[-1] - val_rf[2]),
          "gains computed (100% minus 60% of the data)", "gain = val[-1] - val[2]")
    check(more_data_needed == {"lr": bool(lr_gain > 0.01), "rf": bool(rf_gain > 0.01)} and not more_data_needed["lr"] and not more_data_needed["rf"],
          "with this data neither model gains more than 0.01 by adding data beyond 60%: the limit is not the quantity", "compare the two gains with 0.01")
    check(sufficient_fraction["lr"] < sufficient_fraction["rf"] and val_lr[-1] > val_rf[-1],
          "the logistic needs less data and its final validation is better than the forest's", "the forest needs more data and stays below the logistic")
done()

---
The commented solution is in `solutions/D_interpret/D2_validation_and_learning_curves.ipynb`. Open it after you pass the check, or after you have really tried.